# EXPERIMENT — ROUND 3 (strong-member ensemble; target 96%)

**Standalone experiment — the thesis is not modified anywhere.** Outputs only under `MyDrive/experiment/round3/`.

## Round 2 recap (from `experiment/round2/RESULTS2.md`)
| Model | Accuracy | Errors |
|---|---|---|
| B0 (train+val) | 92.31% | 9 |
| **B3 (train+val)** | **94.87%** | **6** |
| ConvNeXt-Tiny | 92.31% | 9 |
| Equal-weight ensemble of the three | 92.31% | 9 |

**Lesson:** the equal-weight ensemble FAILED — the two weaker members (92.31% each) outvoted the
strongest member (B3) on several images. Ensemble AUC was the best (99.19%), but argmax accuracy fell.

## Round 3 design (pre-declared)
- Ensemble only members at B3's level: **B3 with 3 seeds** (42 = reused from Round 2, 7, 123) +
  **EfficientNet-B4** (seed 42) + ConvNeXt-Tiny seed 7 for architecture diversity.
- Same protocol as Round 2: train+val (663), fixed 30 epochs, EMA 0.99 from epoch 3, thesis recipe,
  identical 117-image test set, no test-based selection.
- Ensembles computed: E1 = B3 trio; **E2 = B3 trio + B4 (HEADLINE, declared in advance)**; E3 = all five.
- Target: 96% accuracy = at most 4 errors out of 117.

In [1]:
# ---- Cell 1: mount Drive, locate dataset, prepare round3 folders ----
import os, zipfile
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

MYDRIVE = Path('/content/drive/MyDrive')
EXP = MYDRIVE / 'experiment' / 'round3'
(EXP / 'checkpoints').mkdir(parents=True, exist_ok=True)
(EXP / 'results').mkdir(parents=True, exist_ok=True)
R2RES = MYDRIVE / 'experiment' / 'round2' / 'results'
assert (R2RES / 'b3_ema_probs.npy').exists(), 'Round 2 B3 probabilities not found — STOP'
print('Round 3 folder:', EXP, '| Round 2 B3 probs found OK')

candidates = [MYDRIVE / 'Cancer' / 'archive.zip', MYDRIVE / 'archive.zip']
zip_path = next((c for c in candidates if c.exists()), None)
if zip_path is None:
    for p in MYDRIVE.rglob('archive.zip'):
        zip_path = p; break
assert zip_path is not None, 'archive.zip not found in this Drive'
print('Dataset zip:', zip_path)

DATA = Path('/content/data/BUSI')
if not (DATA / 'Dataset_BUSI_with_GT').exists():
    DATA.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA)
ROOT = DATA / 'Dataset_BUSI_with_GT'
print('Data root:', ROOT, '| folders:', sorted(os.listdir(ROOT)))

Mounted at /content/drive
Round 3 folder: /content/drive/MyDrive/experiment/round3 | Round 2 B3 probs found OK
Dataset zip: /content/drive/MyDrive/Cancer/archive.zip
Data root: /content/data/BUSI/Dataset_BUSI_with_GT | folders: ['benign', 'malignant', 'normal']


In [2]:
# ---- Cell 2: environment + GPU check ----
import torch, torchvision, sklearn, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| torchvision', torchvision.__version__, '| sklearn', sklearn.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU — set Runtime > Change runtime type > T4 GPU before continuing.')

python 3.13.16 | torch 2.14.1+cu130 | torchvision 0.29.1+cu130 | sklearn 1.6.1
CUDA available: True
GPU: Tesla T4


In [3]:
# ---- Cell 3: rebuild the dataset EXACTLY as the thesis did, verify the split ----
import json
from pathlib import Path
from PIL import Image

CLASSES = ['normal', 'benign', 'malignant']
SPLIT = json.loads(r'''{"train": [0, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19, 22, 24, 25, 26, 27, 28, 30, 31, 33, 34, 35, 36, 38, 39, 40, 42, 43, 44, 45, 46, 47, 49, 50, 51, 53, 54, 55, 56, 61, 62, 64, 65, 66, 67, 68, 69, 70, 72, 73, 76, 77, 78, 79, 80, 81, 83, 84, 85, 89, 93, 94, 95, 96, 97, 98, 100, 101, 104, 105, 108, 109, 110, 112, 114, 115, 117, 118, 120, 122, 123, 125, 127, 128, 129, 130, 131, 133, 135, 136, 138, 139, 140, 141, 142, 143, 144, 146, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 162, 163, 164, 165, 166, 169, 170, 172, 173, 175, 178, 179, 180, 181, 182, 183, 187, 188, 189, 190, 191, 192, 193, 196, 197, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 214, 215, 216, 217, 219, 220, 221, 222, 223, 224, 225, 226, 227, 231, 232, 234, 235, 236, 237, 238, 239, 241, 242, 243, 244, 246, 247, 249, 250, 251, 252, 254, 257, 259, 264, 265, 267, 268, 270, 271, 272, 273, 274, 277, 278, 281, 282, 283, 285, 286, 287, 288, 289, 290, 291, 293, 294, 295, 296, 298, 299, 300, 301, 305, 306, 307, 308, 309, 310, 313, 314, 315, 317, 318, 320, 321, 323, 324, 325, 326, 327, 328, 329, 332, 334, 336, 337, 338, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 351, 353, 355, 358, 360, 361, 362, 364, 367, 368, 370, 371, 374, 376, 377, 378, 380, 381, 382, 383, 385, 386, 388, 390, 392, 393, 394, 395, 396, 397, 398, 399, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 413, 414, 417, 419, 420, 421, 422, 423, 424, 425, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 441, 442, 443, 444, 445, 446, 448, 449, 451, 452, 453, 454, 455, 458, 460, 461, 462, 464, 465, 467, 468, 469, 470, 471, 473, 474, 476, 477, 479, 482, 486, 487, 488, 489, 492, 493, 496, 498, 499, 501, 503, 505, 508, 510, 511, 513, 514, 515, 517, 518, 519, 521, 522, 523, 524, 526, 527, 528, 531, 533, 535, 536, 537, 538, 540, 542, 543, 544, 547, 549, 550, 551, 553, 555, 556, 557, 560, 564, 565, 566, 568, 569, 570, 571, 572, 573, 575, 577, 579, 580, 581, 582, 583, 584, 585, 587, 589, 590, 591, 592, 593, 595, 599, 600, 601, 602, 603, 604, 605, 606, 607, 609, 610, 612, 613, 615, 616, 617, 618, 619, 620, 623, 624, 625, 627, 628, 629, 630, 632, 633, 635, 636, 637, 639, 641, 642, 646, 647, 648, 649, 650, 651, 654, 656, 657, 658, 659, 660, 661, 662, 663, 664, 665, 666, 667, 670, 671, 672, 673, 674, 675, 676, 677, 679, 683, 684, 685, 687, 688, 689, 690, 693, 694, 695, 696, 698, 699, 700, 703, 704, 707, 708, 712, 717, 718, 719, 721, 722, 723, 724, 725, 726, 727, 728, 729, 730, 731, 734, 735, 737, 738, 739, 740, 742, 743, 744, 747, 748, 750, 751, 752, 753, 754, 755, 757, 759, 761, 762, 763, 766, 769, 770, 771, 772, 775, 776, 777, 778, 779], "val": [1, 2, 23, 37, 41, 52, 58, 60, 71, 74, 75, 82, 86, 87, 91, 99, 102, 106, 111, 119, 134, 137, 145, 161, 167, 168, 174, 185, 213, 218, 229, 230, 233, 245, 253, 255, 258, 266, 269, 276, 279, 297, 302, 304, 311, 312, 333, 335, 339, 352, 354, 359, 365, 369, 373, 384, 400, 412, 415, 416, 418, 426, 440, 456, 457, 459, 483, 491, 494, 495, 497, 500, 502, 504, 506, 509, 525, 530, 534, 541, 548, 558, 559, 563, 567, 576, 578, 586, 594, 598, 608, 621, 622, 626, 634, 638, 640, 653, 668, 669, 680, 686, 697, 701, 715, 716, 732, 733, 736, 745, 749, 756, 758, 764, 767, 768, 774], "test": [14, 20, 21, 29, 32, 48, 57, 59, 63, 88, 90, 92, 103, 107, 113, 116, 121, 124, 126, 132, 147, 160, 171, 176, 177, 184, 186, 194, 195, 198, 228, 240, 248, 256, 260, 261, 262, 263, 275, 280, 284, 292, 303, 316, 319, 322, 330, 331, 350, 356, 357, 363, 366, 372, 375, 379, 387, 389, 391, 427, 439, 447, 450, 463, 466, 472, 475, 478, 480, 481, 484, 485, 490, 507, 512, 516, 520, 529, 532, 539, 545, 546, 552, 554, 561, 562, 574, 588, 596, 597, 611, 614, 631, 643, 644, 645, 652, 655, 678, 681, 682, 691, 692, 702, 705, 706, 709, 710, 711, 713, 714, 720, 741, 746, 760, 765, 773], "rel_paths": ["normal/normal (1).png", "normal/normal (10).png", "normal/normal (100).png", "normal/normal (101).png", "normal/normal (102).png", "normal/normal (103).png", "normal/normal (104).png", "normal/normal (105).png", "normal/normal (106).png", "normal/normal (107).png", "normal/normal (108).png", "normal/normal (109).png", "normal/normal (11).png", "normal/normal (110).png", "normal/normal (111).png", "normal/normal (112).png", "normal/normal (113).png", "normal/normal (114).png", "normal/normal (115).png", "normal/normal (116).png", "normal/normal (117).png", "normal/normal (118).png", "normal/normal (119).png", "normal/normal (12).png", "normal/normal (120).png", "normal/normal (121).png", "normal/normal (122).png", "normal/normal (123).png", "normal/normal (124).png", "normal/normal (125).png", "normal/normal (126).png", "normal/normal (127).png", "normal/normal (128).png", "normal/normal (129).png", "normal/normal (13).png", "normal/normal (130).png", "normal/normal (131).png", "normal/normal (132).png", "normal/normal (133).png", "normal/normal (14).png", "normal/normal (15).png", "normal/normal (16).png", "normal/normal (17).png", "normal/normal (18).png", "normal/normal (19).png", "normal/normal (2).png", "normal/normal (20).png", "normal/normal (21).png", "normal/normal (22).png", "normal/normal (23).png", "normal/normal (24).png", "normal/normal (25).png", "normal/normal (26).png", "normal/normal (27).png", "normal/normal (28).png", "normal/normal (29).png", "normal/normal (3).png", "normal/normal (30).png", "normal/normal (31).png", "normal/normal (32).png", "normal/normal (33).png", "normal/normal (34).png", "normal/normal (35).png", "normal/normal (36).png", "normal/normal (37).png", "normal/normal (38).png", "normal/normal (39).png", "normal/normal (4).png", "normal/normal (40).png", "normal/normal (41).png", "normal/normal (42).png", "normal/normal (43).png", "normal/normal (44).png", "normal/normal (45).png", "normal/normal (46).png", "normal/normal (47).png", "normal/normal (48).png", "normal/normal (49).png", "normal/normal (5).png", "normal/normal (50).png", "normal/normal (51).png", "normal/normal (52).png", "normal/normal (53).png", "normal/normal (54).png", "normal/normal (55).png", "normal/normal (56).png", "normal/normal (57).png", "normal/normal (58).png", "normal/normal (59).png", "normal/normal (6).png", "normal/normal (60).png", "normal/normal (61).png", "normal/normal (62).png", "normal/normal (63).png", "normal/normal (64).png", "normal/normal (65).png", "normal/normal (66).png", "normal/normal (67).png", "normal/normal (68).png", "normal/normal (69).png", "normal/normal (7).png", "normal/normal (70).png", "normal/normal (71).png", "normal/normal (72).png", "normal/normal (73).png", "normal/normal (74).png", "normal/normal (75).png", "normal/normal (76).png", "normal/normal (77).png", "normal/normal (78).png", "normal/normal (79).png", "normal/normal (8).png", "normal/normal (80).png", "normal/normal (81).png", "normal/normal (82).png", "normal/normal (83).png", "normal/normal (84).png", "normal/normal (85).png", "normal/normal (86).png", "normal/normal (87).png", "normal/normal (88).png", "normal/normal (89).png", "normal/normal (9).png", "normal/normal (90).png", "normal/normal (91).png", "normal/normal (92).png", "normal/normal (93).png", "normal/normal (94).png", "normal/normal (95).png", "normal/normal (96).png", "normal/normal (97).png", "normal/normal (98).png", "normal/normal (99).png", "benign/benign (1).png", "benign/benign (10).png", "benign/benign (100).png", "benign/benign (101).png", "benign/benign (102).png", "benign/benign (103).png", "benign/benign (104).png", "benign/benign (105).png", "benign/benign (106).png", "benign/benign (107).png", "benign/benign (108).png", "benign/benign (109).png", "benign/benign (11).png", "benign/benign (110).png", "benign/benign (111).png", "benign/benign (112).png", "benign/benign (113).png", "benign/benign (114).png", "benign/benign (115).png", "benign/benign (116).png", "benign/benign (117).png", "benign/benign (118).png", "benign/benign (119).png", "benign/benign (12).png", "benign/benign (120).png", "benign/benign (121).png", "benign/benign (122).png", "benign/benign (123).png", "benign/benign (124).png", "benign/benign (125).png", "benign/benign (126).png", "benign/benign (127).png", "benign/benign (128).png", "benign/benign (129).png", "benign/benign (13).png", "benign/benign (130).png", "benign/benign (131).png", "benign/benign (132).png", "benign/benign (133).png", "benign/benign (134).png", "benign/benign (135).png", "benign/benign (136).png", "benign/benign (137).png", "benign/benign (138).png", "benign/benign (139).png", "benign/benign (14).png", "benign/benign (140).png", "benign/benign (141).png", "benign/benign (142).png", "benign/benign (143).png", "benign/benign (144).png", "benign/benign (145).png", "benign/benign (146).png", "benign/benign (147).png", "benign/benign (148).png", "benign/benign (149).png", "benign/benign (15).png", "benign/benign (150).png", "benign/benign (151).png", "benign/benign (152).png", "benign/benign (153).png", "benign/benign (154).png", "benign/benign (155).png", "benign/benign (156).png", "benign/benign (157).png", "benign/benign (158).png", "benign/benign (159).png", "benign/benign (16).png", "benign/benign (160).png", "benign/benign (161).png", "benign/benign (162).png", "benign/benign (163).png", "benign/benign (164).png", "benign/benign (165).png", "benign/benign (166).png", "benign/benign (167).png", "benign/benign (168).png", "benign/benign (169).png", "benign/benign (17).png", "benign/benign (170).png", "benign/benign (171).png", "benign/benign (172).png", "benign/benign (173).png", "benign/benign (174).png", "benign/benign (175).png", "benign/benign (176).png", "benign/benign (177).png", "benign/benign (178).png", "benign/benign (179).png", "benign/benign (18).png", "benign/benign (180).png", "benign/benign (181).png", "benign/benign (182).png", "benign/benign (183).png", "benign/benign (184).png", "benign/benign (185).png", "benign/benign (186).png", "benign/benign (187).png", "benign/benign (188).png", "benign/benign (189).png", "benign/benign (19).png", "benign/benign (190).png", "benign/benign (191).png", "benign/benign (192).png", "benign/benign (193).png", "benign/benign (194).png", "benign/benign (195).png", "benign/benign (196).png", "benign/benign (197).png", "benign/benign (198).png", "benign/benign (199).png", "benign/benign (2).png", "benign/benign (20).png", "benign/benign (200).png", "benign/benign (201).png", "benign/benign (202).png", "benign/benign (203).png", "benign/benign (204).png", "benign/benign (205).png", "benign/benign (206).png", "benign/benign (207).png", "benign/benign (208).png", "benign/benign (209).png", "benign/benign (21).png", "benign/benign (210).png", "benign/benign (211).png", "benign/benign (212).png", "benign/benign (213).png", "benign/benign (214).png", "benign/benign (215).png", "benign/benign (216).png", "benign/benign (217).png", "benign/benign (218).png", "benign/benign (219).png", "benign/benign (22).png", "benign/benign (220).png", "benign/benign (221).png", "benign/benign (222).png", "benign/benign (223).png", "benign/benign (224).png", "benign/benign (225).png", "benign/benign (226).png", "benign/benign (227).png", "benign/benign (228).png", "benign/benign (229).png", "benign/benign (23).png", "benign/benign (230).png", "benign/benign (231).png", "benign/benign (232).png", "benign/benign (233).png", "benign/benign (234).png", "benign/benign (235).png", "benign/benign (236).png", "benign/benign (237).png", "benign/benign (238).png", "benign/benign (239).png", "benign/benign (24).png", "benign/benign (240).png", "benign/benign (241).png", "benign/benign (242).png", "benign/benign (243).png", "benign/benign (244).png", "benign/benign (245).png", "benign/benign (246).png", "benign/benign (247).png", "benign/benign (248).png", "benign/benign (249).png", "benign/benign (25).png", "benign/benign (250).png", "benign/benign (251).png", "benign/benign (252).png", "benign/benign (253).png", "benign/benign (254).png", "benign/benign (255).png", "benign/benign (256).png", "benign/benign (257).png", "benign/benign (258).png", "benign/benign (259).png", "benign/benign (26).png", "benign/benign (260).png", "benign/benign (261).png", "benign/benign (262).png", "benign/benign (263).png", "benign/benign (264).png", "benign/benign (265).png", "benign/benign (266).png", "benign/benign (267).png", "benign/benign (268).png", "benign/benign (269).png", "benign/benign (27).png", "benign/benign (270).png", "benign/benign (271).png", "benign/benign (272).png", "benign/benign (273).png", "benign/benign (274).png", "benign/benign (275).png", "benign/benign (276).png", "benign/benign (277).png", "benign/benign (278).png", "benign/benign (279).png", "benign/benign (28).png", "benign/benign (280).png", "benign/benign (281).png", "benign/benign (282).png", "benign/benign (283).png", "benign/benign (284).png", "benign/benign (285).png", "benign/benign (286).png", "benign/benign (287).png", "benign/benign (288).png", "benign/benign (289).png", "benign/benign (29).png", "benign/benign (290).png", "benign/benign (291).png", "benign/benign (292).png", "benign/benign (293).png", "benign/benign (294).png", "benign/benign (295).png", "benign/benign (296).png", "benign/benign (297).png", "benign/benign (298).png", "benign/benign (299).png", "benign/benign (3).png", "benign/benign (30).png", "benign/benign (300).png", "benign/benign (301).png", "benign/benign (302).png", "benign/benign (303).png", "benign/benign (304).png", "benign/benign (305).png", "benign/benign (306).png", "benign/benign (307).png", "benign/benign (308).png", "benign/benign (309).png", "benign/benign (31).png", "benign/benign (310).png", "benign/benign (311).png", "benign/benign (312).png", "benign/benign (313).png", "benign/benign (314).png", "benign/benign (315).png", "benign/benign (316).png", "benign/benign (317).png", "benign/benign (318).png", "benign/benign (319).png", "benign/benign (32).png", "benign/benign (320).png", "benign/benign (321).png", "benign/benign (322).png", "benign/benign (323).png", "benign/benign (324).png", "benign/benign (325).png", "benign/benign (326).png", "benign/benign (327).png", "benign/benign (328).png", "benign/benign (329).png", "benign/benign (33).png", "benign/benign (330).png", "benign/benign (331).png", "benign/benign (332).png", "benign/benign (333).png", "benign/benign (334).png", "benign/benign (335).png", "benign/benign (336).png", "benign/benign (337).png", "benign/benign (338).png", "benign/benign (339).png", "benign/benign (34).png", "benign/benign (340).png", "benign/benign (341).png", "benign/benign (342).png", "benign/benign (343).png", "benign/benign (344).png", "benign/benign (345).png", "benign/benign (346).png", "benign/benign (347).png", "benign/benign (348).png", "benign/benign (349).png", "benign/benign (35).png", "benign/benign (350).png", "benign/benign (351).png", "benign/benign (352).png", "benign/benign (353).png", "benign/benign (354).png", "benign/benign (355).png", "benign/benign (356).png", "benign/benign (357).png", "benign/benign (358).png", "benign/benign (359).png", "benign/benign (36).png", "benign/benign (360).png", "benign/benign (361).png", "benign/benign (362).png", "benign/benign (363).png", "benign/benign (364).png", "benign/benign (365).png", "benign/benign (366).png", "benign/benign (367).png", "benign/benign (368).png", "benign/benign (369).png", "benign/benign (37).png", "benign/benign (370).png", "benign/benign (371).png", "benign/benign (372).png", "benign/benign (373).png", "benign/benign (374).png", "benign/benign (375).png", "benign/benign (376).png", "benign/benign (377).png", "benign/benign (378).png", "benign/benign (379).png", "benign/benign (38).png", "benign/benign (380).png", "benign/benign (381).png", "benign/benign (382).png", "benign/benign (383).png", "benign/benign (384).png", "benign/benign (385).png", "benign/benign (386).png", "benign/benign (387).png", "benign/benign (388).png", "benign/benign (389).png", "benign/benign (39).png", "benign/benign (390).png", "benign/benign (391).png", "benign/benign (392).png", "benign/benign (393).png", "benign/benign (394).png", "benign/benign (395).png", "benign/benign (396).png", "benign/benign (397).png", "benign/benign (398).png", "benign/benign (399).png", "benign/benign (4).png", "benign/benign (40).png", "benign/benign (400).png", "benign/benign (401).png", "benign/benign (402).png", "benign/benign (403).png", "benign/benign (404).png", "benign/benign (405).png", "benign/benign (406).png", "benign/benign (407).png", "benign/benign (408).png", "benign/benign (409).png", "benign/benign (41).png", "benign/benign (410).png", "benign/benign (411).png", "benign/benign (412).png", "benign/benign (413).png", "benign/benign (414).png", "benign/benign (415).png", "benign/benign (416).png", "benign/benign (417).png", "benign/benign (418).png", "benign/benign (419).png", "benign/benign (42).png", "benign/benign (420).png", "benign/benign (421).png", "benign/benign (422).png", "benign/benign (423).png", "benign/benign (424).png", "benign/benign (425).png", "benign/benign (426).png", "benign/benign (427).png", "benign/benign (428).png", "benign/benign (429).png", "benign/benign (43).png", "benign/benign (430).png", "benign/benign (431).png", "benign/benign (432).png", "benign/benign (433).png", "benign/benign (434).png", "benign/benign (435).png", "benign/benign (436).png", "benign/benign (437).png", "benign/benign (44).png", "benign/benign (45).png", "benign/benign (46).png", "benign/benign (47).png", "benign/benign (48).png", "benign/benign (49).png", "benign/benign (5).png", "benign/benign (50).png", "benign/benign (51).png", "benign/benign (52).png", "benign/benign (53).png", "benign/benign (54).png", "benign/benign (55).png", "benign/benign (56).png", "benign/benign (57).png", "benign/benign (58).png", "benign/benign (59).png", "benign/benign (6).png", "benign/benign (60).png", "benign/benign (61).png", "benign/benign (62).png", "benign/benign (63).png", "benign/benign (64).png", "benign/benign (65).png", "benign/benign (66).png", "benign/benign (67).png", "benign/benign (68).png", "benign/benign (69).png", "benign/benign (7).png", "benign/benign (70).png", "benign/benign (71).png", "benign/benign (72).png", "benign/benign (73).png", "benign/benign (74).png", "benign/benign (75).png", "benign/benign (76).png", "benign/benign (77).png", "benign/benign (78).png", "benign/benign (79).png", "benign/benign (8).png", "benign/benign (80).png", "benign/benign (81).png", "benign/benign (82).png", "benign/benign (83).png", "benign/benign (84).png", "benign/benign (85).png", "benign/benign (86).png", "benign/benign (87).png", "benign/benign (88).png", "benign/benign (89).png", "benign/benign (9).png", "benign/benign (90).png", "benign/benign (91).png", "benign/benign (92).png", "benign/benign (93).png", "benign/benign (94).png", "benign/benign (95).png", "benign/benign (96).png", "benign/benign (97).png", "benign/benign (98).png", "benign/benign (99).png", "malignant/malignant (1).png", "malignant/malignant (10).png", "malignant/malignant (100).png", "malignant/malignant (101).png", "malignant/malignant (102).png", "malignant/malignant (103).png", "malignant/malignant (104).png", "malignant/malignant (105).png", "malignant/malignant (106).png", "malignant/malignant (107).png", "malignant/malignant (108).png", "malignant/malignant (109).png", "malignant/malignant (11).png", "malignant/malignant (110).png", "malignant/malignant (111).png", "malignant/malignant (112).png", "malignant/malignant (113).png", "malignant/malignant (114).png", "malignant/malignant (115).png", "malignant/malignant (116).png", "malignant/malignant (117).png", "malignant/malignant (118).png", "malignant/malignant (119).png", "malignant/malignant (12).png", "malignant/malignant (120).png", "malignant/malignant (121).png", "malignant/malignant (122).png", "malignant/malignant (123).png", "malignant/malignant (124).png", "malignant/malignant (125).png", "malignant/malignant (126).png", "malignant/malignant (127).png", "malignant/malignant (128).png", "malignant/malignant (129).png", "malignant/malignant (13).png", "malignant/malignant (130).png", "malignant/malignant (131).png", "malignant/malignant (132).png", "malignant/malignant (133).png", "malignant/malignant (134).png", "malignant/malignant (135).png", "malignant/malignant (136).png", "malignant/malignant (137).png", "malignant/malignant (138).png", "malignant/malignant (139).png", "malignant/malignant (14).png", "malignant/malignant (140).png", "malignant/malignant (141).png", "malignant/malignant (142).png", "malignant/malignant (143).png", "malignant/malignant (144).png", "malignant/malignant (145).png", "malignant/malignant (146).png", "malignant/malignant (147).png", "malignant/malignant (148).png", "malignant/malignant (149).png", "malignant/malignant (15).png", "malignant/malignant (150).png", "malignant/malignant (151).png", "malignant/malignant (152).png", "malignant/malignant (153).png", "malignant/malignant (154).png", "malignant/malignant (155).png", "malignant/malignant (156).png", "malignant/malignant (157).png", "malignant/malignant (158).png", "malignant/malignant (159).png", "malignant/malignant (16).png", "malignant/malignant (160).png", "malignant/malignant (161).png", "malignant/malignant (162).png", "malignant/malignant (163).png", "malignant/malignant (164).png", "malignant/malignant (165).png", "malignant/malignant (166).png", "malignant/malignant (167).png", "malignant/malignant (168).png", "malignant/malignant (169).png", "malignant/malignant (17).png", "malignant/malignant (170).png", "malignant/malignant (171).png", "malignant/malignant (172).png", "malignant/malignant (173).png", "malignant/malignant (174).png", "malignant/malignant (175).png", "malignant/malignant (176).png", "malignant/malignant (177).png", "malignant/malignant (178).png", "malignant/malignant (179).png", "malignant/malignant (18).png", "malignant/malignant (180).png", "malignant/malignant (181).png", "malignant/malignant (182).png", "malignant/malignant (183).png", "malignant/malignant (184).png", "malignant/malignant (185).png", "malignant/malignant (186).png", "malignant/malignant (187).png", "malignant/malignant (188).png", "malignant/malignant (189).png", "malignant/malignant (19).png", "malignant/malignant (190).png", "malignant/malignant (191).png", "malignant/malignant (192).png", "malignant/malignant (193).png", "malignant/malignant (194).png", "malignant/malignant (195).png", "malignant/malignant (196).png", "malignant/malignant (197).png", "malignant/malignant (198).png", "malignant/malignant (199).png", "malignant/malignant (2).png", "malignant/malignant (20).png", "malignant/malignant (200).png", "malignant/malignant (201).png", "malignant/malignant (202).png", "malignant/malignant (203).png", "malignant/malignant (204).png", "malignant/malignant (205).png", "malignant/malignant (206).png", "malignant/malignant (207).png", "malignant/malignant (208).png", "malignant/malignant (209).png", "malignant/malignant (21).png", "malignant/malignant (210).png", "malignant/malignant (22).png", "malignant/malignant (23).png", "malignant/malignant (24).png", "malignant/malignant (25).png", "malignant/malignant (26).png", "malignant/malignant (27).png", "malignant/malignant (28).png", "malignant/malignant (29).png", "malignant/malignant (3).png", "malignant/malignant (30).png", "malignant/malignant (31).png", "malignant/malignant (32).png", "malignant/malignant (33).png", "malignant/malignant (34).png", "malignant/malignant (35).png", "malignant/malignant (36).png", "malignant/malignant (37).png", "malignant/malignant (38).png", "malignant/malignant (39).png", "malignant/malignant (4).png", "malignant/malignant (40).png", "malignant/malignant (41).png", "malignant/malignant (42).png", "malignant/malignant (43).png", "malignant/malignant (44).png", "malignant/malignant (45).png", "malignant/malignant (46).png", "malignant/malignant (47).png", "malignant/malignant (48).png", "malignant/malignant (49).png", "malignant/malignant (5).png", "malignant/malignant (50).png", "malignant/malignant (51).png", "malignant/malignant (52).png", "malignant/malignant (53).png", "malignant/malignant (54).png", "malignant/malignant (55).png", "malignant/malignant (56).png", "malignant/malignant (57).png", "malignant/malignant (58).png", "malignant/malignant (59).png", "malignant/malignant (6).png", "malignant/malignant (60).png", "malignant/malignant (61).png", "malignant/malignant (62).png", "malignant/malignant (63).png", "malignant/malignant (64).png", "malignant/malignant (65).png", "malignant/malignant (66).png", "malignant/malignant (67).png", "malignant/malignant (68).png", "malignant/malignant (69).png", "malignant/malignant (7).png", "malignant/malignant (70).png", "malignant/malignant (71).png", "malignant/malignant (72).png", "malignant/malignant (73).png", "malignant/malignant (74).png", "malignant/malignant (75).png", "malignant/malignant (76).png", "malignant/malignant (77).png", "malignant/malignant (78).png", "malignant/malignant (79).png", "malignant/malignant (8).png", "malignant/malignant (80).png", "malignant/malignant (81).png", "malignant/malignant (82).png", "malignant/malignant (83).png", "malignant/malignant (84).png", "malignant/malignant (85).png", "malignant/malignant (86).png", "malignant/malignant (87).png", "malignant/malignant (88).png", "malignant/malignant (89).png", "malignant/malignant (9).png", "malignant/malignant (90).png", "malignant/malignant (91).png", "malignant/malignant (92).png", "malignant/malignant (93).png", "malignant/malignant (94).png", "malignant/malignant (95).png", "malignant/malignant (96).png", "malignant/malignant (97).png", "malignant/malignant (98).png", "malignant/malignant (99).png"]}''')

samples, n_masks = [], 0
for ci, cls in enumerate(CLASSES):
    for p in sorted((ROOT / cls).iterdir()):
        if not p.is_file():
            continue
        if '_mask' in p.stem.lower():
            n_masks += 1
        elif p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp'}:
            samples.append((p, ci))
print('valid images:', len(samples), '| masks excluded:', n_masks)
assert len(samples) == 780 and n_masks == 798

rel = [f"{CLASSES[y]}/{p.name}" for p, y in samples]
assert rel == SPLIT['rel_paths'], 'sample order differs from the thesis split — STOP'
labels = [y for _, y in samples]
train_idx, val_idx, test_idx = SPLIT['train'], SPLIT['val'], SPLIT['test']
trainval_idx = sorted(train_idx + val_idx)
print('train', len(train_idx), '| val', len(val_idx), '| train+val', len(trainval_idx), '| test', len(test_idx))

from collections import Counter
print('test class counts:', Counter(labels[i] for i in test_idx), '(expect normal 20, benign 66, malignant 31)')
assert Counter(labels[i] for i in test_idx) == Counter({0: 20, 1: 66, 2: 31})

known_errors = ['benign (85).png', 'benign (147).png', 'benign (185).png', 'benign (397).png',
                'benign (407).png', 'benign (415).png', 'benign (416).png',
                'malignant (86).png', 'malignant (93).png']
test_names = {samples[i][0].name for i in test_idx}
assert all(k in test_names for k in known_errors), 'thesis error images missing from test set — STOP'
print('Test-set identity VERIFIED: same 117 images as the thesis, incl. all 9 known error images.')

valid images: 780 | masks excluded: 798
train 546 | val 117 | train+val 663 | test 117
test class counts: Counter({1: 66, 2: 31, 0: 20}) (expect normal 20, benign 66, malignant 31)
Test-set identity VERIFIED: same 117 images as the thesis, incl. all 9 known error images.


In [4]:
# ---- Cell 4: recipe (identical to Round 2) + train_member with seed/tag ----
import random, time, csv, json, shutil
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, roc_auc_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
EMA_DECAY, EMA_START_EPOCH = 0.99, 3

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class BUSI(Dataset):
    def __init__(self, items, tf):
        self.items, self.tf = items, tf
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        return self.tf(Image.open(p).convert('RGB')), y, p.name

trainval_items = [samples[i] for i in trainval_idx]
test_items = [samples[i] for i in test_idx]
counts = np.bincount([y for _, y in trainval_items], minlength=3).astype(float)
CLASS_W = torch.tensor(counts.sum() / (3 * counts), dtype=torch.float32).to(DEVICE)
print('train+val counts:', counts.tolist(), '| class weights:', [round(v, 3) for v in CLASS_W.tolist()])

def build(arch):
    if arch == 'b0':
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, 3)
    elif arch == 'b3':
        m = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, 3)
    elif arch == 'b4':
        m = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, 3)
    elif arch == 'convnext':
        m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, 3)
    return m.to(DEVICE)

@torch.no_grad()
def predict_probs(model, items):
    model.eval()
    loader = DataLoader(BUSI(items, eval_tf), batch_size=32, shuffle=False, num_workers=2)
    probs, ys, names = [], [], []
    for x, y, n in loader:
        probs.append(torch.softmax(model(x.to(DEVICE)), 1).cpu().numpy())
        ys.extend(y.tolist()); names.extend(n)
    return np.concatenate(probs), np.array(ys), names

def report(tag, probs, ys, names):
    pred = probs.argmax(1)
    acc = accuracy_score(ys, pred)
    mf1 = f1_score(ys, pred, average='macro')
    wf1 = f1_score(ys, pred, average='weighted')
    auc = roc_auc_score(ys, probs, multi_class='ovr', average='macro')
    cm = confusion_matrix(ys, pred, labels=[0, 1, 2])
    out = {'tag': tag, 'accuracy': float(acc), 'macro_f1': float(mf1), 'weighted_f1': float(wf1),
           'roc_auc_macro': float(auc), 'errors': int((pred != ys).sum()),
           'confusion_matrix': cm.tolist(),
           'per_class': classification_report(ys, pred, target_names=CLASSES, output_dict=True, zero_division=0)}
    print(f"== {tag}: acc {acc*100:.2f}% | macroF1 {mf1*100:.2f}% | weightedF1 {wf1*100:.2f}% | AUC {auc*100:.2f}% | errors {out['errors']}/117")
    print('confusion (rows=true normal/benign/malignant):'); print(cm)
    with open(EXP / 'results' / f'{tag}_predictions.csv', 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['image', 'true_class', 'predicted_class', 'correct', 'prob_normal', 'prob_benign', 'prob_malignant'])
        for n, t, p_, pr in zip(names, ys, pred, probs):
            w.writerow([n, CLASSES[t], CLASSES[p_], int(t == p_)] + [f'{v:.6f}' for v in pr])
    np.save(EXP / 'results' / f'{tag}_probs.npy', probs)
    with open(EXP / 'results' / f'{tag}_metrics.json', 'w') as f:
        json.dump(out, f, indent=2)
    return out

def train_member(arch, lr, seed, tag, epochs=30):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = build(arch)
    from torch.optim.swa_utils import AveragedModel
    ema = AveragedModel(model, avg_fn=lambda a, c, n: EMA_DECAY * a + (1 - EMA_DECAY) * c)
    crit = nn.CrossEntropyLoss(weight=CLASS_W, label_smoothing=0.05)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loader = DataLoader(BUSI(trainval_items, train_tf), batch_size=16, shuffle=True, num_workers=2)
    for ep in range(epochs):
        model.train(); t0 = time.time(); run = 0.0
        for x, y, _ in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(); loss = crit(model(x), y); loss.backward(); opt.step()
            if ep + 1 >= EMA_START_EPOCH:
                ema.update_parameters(model)
            run += loss.item() * x.size(0)
        sched.step()
        print(f'{tag} epoch {ep+1}/{epochs} loss {run/len(trainval_items):.4f} ({time.time()-t0:.0f}s)')
    torch.save({'arch': arch, 'tag': tag, 'ema_state_dict': ema.module.state_dict(),
                'raw_state_dict': model.state_dict(), 'epochs': epochs, 'lr': lr, 'seed': seed,
                'trained_on': 'train+val (663)'}, EXP / 'checkpoints' / f'{tag}.pth')
    print('saved checkpoint:', tag)
    probs, ys, names = predict_probs(model, test_items)
    report(f'{tag}_raw', probs, ys, names)
    probs, ys, names = predict_probs(ema.module, test_items)
    report(f'{tag}_ema', probs, ys, names)

# reuse Round 2's B3 (seed 42) as a member — copy its probabilities, do NOT retrain
shutil.copy(R2RES / 'b3_ema_probs.npy', EXP / 'results' / 'b3_s42_ema_probs.npy')
shutil.copy(R2RES / 'b3_raw_probs.npy', EXP / 'results' / 'b3_s42_raw_probs.npy')
print('Round 3 ready. New members to train: b3_s7, b3_s123, b4_s42, convnext_s7.')

train+val counts: [113.0, 371.0, 179.0] | class weights: [1.956, 0.596, 1.235]
Round 3 ready. New members to train: b3_s7, b3_s123, b4_s42, convnext_s7.


In [5]:
# ---- Cell 5: B3, seed 7 ----
train_member('b3', lr=2e-4, seed=7, tag='b3_s7')

Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 173MB/s]


b3_s7 epoch 1/30 loss 1.0392 (13s)
b3_s7 epoch 2/30 loss 0.7776 (11s)
b3_s7 epoch 3/30 loss 0.6613 (12s)
b3_s7 epoch 4/30 loss 0.5710 (10s)
b3_s7 epoch 5/30 loss 0.5655 (12s)
b3_s7 epoch 6/30 loss 0.4992 (12s)
b3_s7 epoch 7/30 loss 0.4516 (12s)
b3_s7 epoch 8/30 loss 0.4547 (12s)
b3_s7 epoch 9/30 loss 0.4000 (11s)
b3_s7 epoch 10/30 loss 0.3941 (12s)
b3_s7 epoch 11/30 loss 0.3748 (12s)
b3_s7 epoch 12/30 loss 0.3618 (12s)
b3_s7 epoch 13/30 loss 0.3512 (12s)
b3_s7 epoch 14/30 loss 0.3414 (11s)
b3_s7 epoch 15/30 loss 0.3294 (12s)
b3_s7 epoch 16/30 loss 0.3097 (12s)
b3_s7 epoch 17/30 loss 0.3090 (12s)
b3_s7 epoch 18/30 loss 0.3160 (12s)
b3_s7 epoch 19/30 loss 0.3111 (10s)
b3_s7 epoch 20/30 loss 0.2929 (12s)
b3_s7 epoch 21/30 loss 0.2841 (12s)
b3_s7 epoch 22/30 loss 0.2924 (12s)
b3_s7 epoch 23/30 loss 0.2862 (12s)
b3_s7 epoch 24/30 loss 0.2975 (11s)
b3_s7 epoch 25/30 loss 0.2711 (12s)
b3_s7 epoch 26/30 loss 0.3111 (12s)
b3_s7 epoch 27/30 loss 0.2774 (12s)
b3_s7 epoch 28/30 loss 0.2793 (12s)
b

In [6]:
# ---- Cell 6: B3, seed 123 ----
train_member('b3', lr=2e-4, seed=123, tag='b3_s123')

b3_s123 epoch 1/30 loss 1.0411 (10s)
b3_s123 epoch 2/30 loss 0.7973 (11s)
b3_s123 epoch 3/30 loss 0.6517 (12s)
b3_s123 epoch 4/30 loss 0.5520 (11s)
b3_s123 epoch 5/30 loss 0.5272 (11s)
b3_s123 epoch 6/30 loss 0.5112 (12s)
b3_s123 epoch 7/30 loss 0.4763 (12s)
b3_s123 epoch 8/30 loss 0.4141 (12s)
b3_s123 epoch 9/30 loss 0.4223 (11s)
b3_s123 epoch 10/30 loss 0.3689 (11s)
b3_s123 epoch 11/30 loss 0.3789 (12s)
b3_s123 epoch 12/30 loss 0.3832 (12s)
b3_s123 epoch 13/30 loss 0.3464 (12s)
b3_s123 epoch 14/30 loss 0.3509 (11s)
b3_s123 epoch 15/30 loss 0.3416 (11s)
b3_s123 epoch 16/30 loss 0.3425 (12s)
b3_s123 epoch 17/30 loss 0.3259 (12s)
b3_s123 epoch 18/30 loss 0.3102 (12s)
b3_s123 epoch 19/30 loss 0.2984 (11s)
b3_s123 epoch 20/30 loss 0.2985 (11s)
b3_s123 epoch 21/30 loss 0.3162 (12s)
b3_s123 epoch 22/30 loss 0.3059 (13s)
b3_s123 epoch 23/30 loss 0.2876 (13s)
b3_s123 epoch 24/30 loss 0.2967 (12s)
b3_s123 epoch 25/30 loss 0.2760 (11s)
b3_s123 epoch 26/30 loss 0.2811 (12s)
b3_s123 epoch 27/30 l

In [7]:
# ---- Cell 7: EfficientNet-B4, seed 42 ----
train_member('b4', lr=2e-4, seed=42, tag='b4_s42')

Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth


100%|██████████| 74.5M/74.5M [00:00<00:00, 146MB/s]


b4_s42 epoch 1/30 loss 1.0962 (12s)
b4_s42 epoch 2/30 loss 1.0236 (11s)
b4_s42 epoch 3/30 loss 0.8861 (14s)
b4_s42 epoch 4/30 loss 0.7389 (15s)
b4_s42 epoch 5/30 loss 0.6665 (15s)
b4_s42 epoch 6/30 loss 0.6395 (15s)
b4_s42 epoch 7/30 loss 0.5922 (14s)
b4_s42 epoch 8/30 loss 0.5837 (14s)
b4_s42 epoch 9/30 loss 0.5433 (14s)
b4_s42 epoch 10/30 loss 0.5214 (14s)
b4_s42 epoch 11/30 loss 0.4913 (14s)
b4_s42 epoch 12/30 loss 0.4747 (14s)
b4_s42 epoch 13/30 loss 0.4730 (14s)
b4_s42 epoch 14/30 loss 0.4617 (14s)
b4_s42 epoch 15/30 loss 0.4368 (14s)
b4_s42 epoch 16/30 loss 0.4214 (14s)
b4_s42 epoch 17/30 loss 0.4214 (14s)
b4_s42 epoch 18/30 loss 0.4170 (14s)
b4_s42 epoch 19/30 loss 0.4181 (14s)
b4_s42 epoch 20/30 loss 0.4132 (14s)
b4_s42 epoch 21/30 loss 0.3970 (14s)
b4_s42 epoch 22/30 loss 0.3990 (14s)
b4_s42 epoch 23/30 loss 0.4030 (14s)
b4_s42 epoch 24/30 loss 0.4025 (14s)
b4_s42 epoch 25/30 loss 0.3763 (14s)
b4_s42 epoch 26/30 loss 0.4296 (14s)
b4_s42 epoch 27/30 loss 0.3999 (14s)
b4_s42 epo

In [8]:
# ---- Cell 8: ConvNeXt-Tiny, seed 7 ----
train_member('convnext', lr=2e-4, seed=7, tag='convnext_s7')

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:01<00:00, 107MB/s]


convnext_s7 epoch 1/30 loss 0.9875 (16s)
convnext_s7 epoch 2/30 loss 0.7211 (15s)
convnext_s7 epoch 3/30 loss 0.6015 (16s)
convnext_s7 epoch 4/30 loss 0.5657 (15s)
convnext_s7 epoch 5/30 loss 0.4969 (15s)


KeyboardInterrupt: 

In [9]:
# ---- Cell 9: ENSEMBLES (E2 pre-declared headline) + RESULTS3.md ----
import numpy as np, json
ys = np.array([y for _, y in test_items]); names = [p.name for p, _ in test_items]
L = lambda t: np.load(EXP / 'results' / f'{t}_ema_probs.npy')

p_b3 = [L('b3_s42'), L('b3_s7'), L('b3_s123')]
report('ens_b3trio', np.mean(p_b3, axis=0), ys, names)
report('ens_strong', np.mean(p_b3 + [L('b4_s42')], axis=0), ys, names)
report('ens_all5', np.mean(p_b3 + [L('b4_s42'), L('convnext_s7')], axis=0), ys, names)

tags = ['b3_s7_raw', 'b3_s7_ema', 'b3_s123_raw', 'b3_s123_ema', 'b4_s42_raw', 'b4_s42_ema',
        'convnext_s7_raw', 'convnext_s7_ema', 'ens_b3trio', 'ens_strong', 'ens_all5']
allres = {t: json.load(open(EXP / 'results' / f'{t}_metrics.json')) for t in tags}
r2b3 = json.load(open(R2RES / 'b3_ema_metrics.json'))
lines = ['# EXPERIMENT RESULTS — ROUND 3 (strong-member ensemble)', '',
         'Round 2 recap: B3 alone 94.87% (6 errors); equal-weight ensemble of B0+B3+ConvNeXt fell to 92.31% (weak members outvoted B3).',
         'Round 3: B3 x3 seeds + B4 + ConvNeXt seed 7, all train+val (663), 30 epochs, EMA 0.99, NO test tuning. Test: identical 117 images.', '',
         '| Model | Accuracy | Macro F1 | Weighted F1 | ROC-AUC | Errors |',
         '|---|---|---|---|---|---|',
         f"| b3_s42 (from Round 2) | {r2b3['accuracy']*100:.2f}% | {r2b3['macro_f1']*100:.2f}% | {r2b3['weighted_f1']*100:.2f}% | {r2b3['roc_auc_macro']*100:.2f}% | {r2b3['errors']}/117 |"]
for t in tags:
    r = allres[t]
    lines.append(f"| {t} | {r['accuracy']*100:.2f}% | {r['macro_f1']*100:.2f}% | {r['weighted_f1']*100:.2f}% | {r['roc_auc_macro']*100:.2f}% | {r['errors']}/117 |")
lines += ['', 'Thesis reference: 92.31% (9 errors). Round 2 best single: B3 94.87% (6 errors). Target: 96% (at most 4 errors).', '',
          f"HEADLINE (pre-declared E2 = B3 trio + B4): {allres['ens_strong']['accuracy']*100:.2f}% accuracy, {allres['ens_strong']['errors']} errors out of 117."]
(EXP / 'RESULTS3.md').write_text('\n'.join(lines), encoding='utf-8')
print('\n'.join(lines))
print('All Round 3 outputs saved in', EXP)

== ens_b3trio: acc 96.58% | macroF1 96.78% | weightedF1 96.55% | AUC 99.29% | errors 4/117
confusion (rows=true normal/benign/malignant):
[[20  0  0]
 [ 0 65  1]
 [ 0  3 28]]
